## Analytical Notebook – Business Insights

This notebook contains analytical queries and visualizations based on Gold layer data.

In [0]:
%sql
-- Top 10 states by revenue and customer distribution
SELECT customer_state, total_revenue,customer_count
FROM olist_gold.vw_revenue_by_state
ORDER BY total_revenue DESC
LIMIT 10;


customer_state,total_revenue,customer_count
SP,5998226.959999896,41746
RJ,2144379.6900000074,12852
MG,1872257.260000002,11635
RS,890898.5399999988,5466
PR,811156.3799999979,5045
SC,623086.43,3637
BA,616645.8200000023,3380
DF,355141.07999999967,2140
GO,350092.3100000006,2020
ES,325967.55000000016,2033


In [0]:
%sql
-- City-Level Revenue Analysis(deep down analysis)
SELECT 
    customer_city,
    customer_state,
    COUNT(*) as customer_count,
    ROUND(SUM(total_revenue), 2) as total_revenue,
    ROUND(AVG(total_revenue), 2) as avg_revenue_per_customer
FROM olist_gold.customer_metrics
WHERE customer_city IS NOT NULL
GROUP BY customer_city, customer_state
ORDER BY total_revenue DESC
LIMIT 10;

customer_city,customer_state,customer_count,total_revenue,avg_revenue_per_customer
sao paulo,SP,15540,2203373.09,141.81
rio de janeiro,RJ,6882,1161927.36,168.84
belo horizonte,MG,2773,421765.12,152.1
brasilia,DF,2131,354216.78,166.22
curitiba,PR,1521,247392.48,162.65
porto alegre,RS,1379,224731.42,162.97
salvador,BA,1245,218071.5,175.16
campinas,SP,1444,216248.43,149.76
guarulhos,SP,1189,165121.99,138.87
niteroi,RJ,849,139996.99,164.9


In [0]:
%sql
-- Product Performance Analysis - Analyze top 15 product categories by revenue
SELECT *
FROM olist_gold.vw_top_products
LIMIT 15;


product_id,product_category_name,total_orders,total_units_sold,total_revenue,avg_price,total_freight,revenue_rank,category_rank
bb50f2e236e5eea0100680137654686c,beleza_saude,187,187,61245.0,327.51,3594.340000000001,1,1
6cdd53843498f92890544667809f1595,beleza_saude,151,151,52966.6000000001,350.77,4181.26,2,2
d6160fb7873f184099d9bc95e30376af,pcs,35,35,48899.340000000004,1397.12,1426.8400000000001,3,1
d1c427060a0f73f6b889a5c7c61f2ac4,informatica_acessorios,323,323,44417.58000000001,137.52,12959.340000000002,4,1
99a4788cb24856965c36a24e339b6058,cama_mesa_banho,467,467,41193.56000000034,88.21,7690.020000000007,5,1
25c38557cf793876c5abdd5931f922db,bebes,38,38,38907.32000000001,1023.88,1404.6299999999999,6,1
3dd2a17168ec895c781a9191c1e95ad7,informatica_acessorios,255,255,38234.50000000018,149.94,6692.920000000004,7,2
5f504b3a1c75b73d6151be81eb05bdc9,cool_stuff,63,63,37733.899999999994,598.95,3991.909999999998,8,1
53b36df67ebb7c41585e8d54d6772e08,relogios_presentes,306,306,35646.820000000116,116.49,2202.3300000000004,9,1
e0d64dcfaa3b6db5c54ca298ae101d05,relogios_presentes,194,194,31786.820000000043,163.85,3557.2699999999995,10,2


In [0]:
%sql

-- Monthly sales trend (Time-Series Analysis) with growth metrics (last 12 months)
SELECT *
FROM olist_gold.vw_monthly_sales_trend
ORDER BY month DESC
LIMIT 12;


month,total_orders,total_revenue,avg_order_value
2018-10-01T00:00:00.000Z,4,589.67,147.42
2018-09-01T00:00:00.000Z,16,4439.540000000001,295.97
2018-08-01T00:00:00.000Z,6512,1022425.3199999981,152.69
2018-07-01T00:00:00.000Z,6292,1066540.7500000002,163.91
2018-06-01T00:00:00.000Z,6167,1023880.4999999978,159.51
2018-05-01T00:00:00.000Z,6873,1153982.149999995,161.74
2018-04-01T00:00:00.000Z,6939,1160785.4799999935,161.02
2018-03-01T00:00:00.000Z,7211,1159652.1199999924,154.37
2018-02-01T00:00:00.000Z,6728,992463.3400000049,142.76
2018-01-01T00:00:00.000Z,7269,1115004.1800000027,147.45


In [0]:
%sql
-- Payment Behavior Analysis - Understand customer payment preferences
-- Payment type distribution and analysis
SELECT *
FROM olist_gold.vw_payment_distribution 
WHERE payment_type IS NOT NULL
ORDER BY total_revenue DESC;


payment_type,total_orders,total_revenue
credit_card,74304,1.2855972719999513E7
boleto,19191,2957800.1299999673
voucher,3679,361406.2200000023
debit_card,1485,226000.4599999995


In [0]:
%sql
-- Delivery performance by time buckets
SELECT *
FROM olist_gold.vw_delivery_performance
ORDER BY order_count DESC;



delivery_bucket,order_count,avg_delivery_days
Normal,41360,10.646615087040619
Fast,33432,4.966020579085906
Slow,17420,17.500344431687715
Very Slow,12468,31.467977528089886


In [0]:
%sql
-- advanced SQL using window functions and is not part of dashboard logic only for learning.
-- Top 3 products in each category using window functions
WITH ranked_products AS (
    SELECT 
        product_category_name,
        product_id,
        total_orders,
        ROUND(total_revenue, 2) as total_revenue,
        ROUND(avg_price, 2) as avg_price,
        
        -- Rank products within each category
        ROW_NUMBER() OVER (PARTITION BY product_category_name 
                          ORDER BY total_revenue DESC) as rank_in_category
    
    FROM olist_gold.product_performance
    WHERE product_category_name IS NOT NULL
)
SELECT 
    product_category_name,
    product_id,
    total_orders,
    total_revenue,
    avg_price,
    rank_in_category
FROM ranked_products
WHERE rank_in_category <= 3
ORDER BY product_category_name, rank_in_category;

product_category_name,product_id,total_orders,total_revenue,avg_price,rank_in_category
agro_industria_e_comercio,11250b0d4b709fee92441c5f34122aed,21,8699.0,414.24,1
agro_industria_e_comercio,423a6644f0aa529e8828ff1f91003690,18,8043.0,446.83,2
agro_industria_e_comercio,672e757f331900b9deea127a2a7b79fd,14,5679.0,405.64,3
alimentos,73326828aa5efe1ba096223de496f596,53,4375.12,82.55,1
alimentos,89321f94e35fc6d7903d36f74e351d40,114,3236.21,28.39,2
alimentos,ed2067a9c1f79553088a3c67b99a9f97,52,3016.59,58.01,3
alimentos_bebidas,992197904e1d4f0bf3994652373188e4,11,2371.16,215.56,1
alimentos_bebidas,90f97298579cd20412fdcc9b7a2d4b6b,11,1348.0,122.55,2
alimentos_bebidas,84f5c4f480ad6c9998d6a6860f1a2e41,24,987.96,41.16,3
artes,4fe644d766c7566dbc46fb851363cb3b,106,10692.73,100.87,1


Dashboard Queries:
- KPI Summary
- Revenue by state
- Monthly sales trends
- Payment distribution
- Delivery performance

Exploratory / Deep-dive:
- City-level analysis

Learning-only:
- Top products per category (window function)